# Notebook 12: Glass Acoustic Resonator — Garage Prototype Design

**Phase 5 — WCFOMA on a $63 Budget**

Glass eliminates the fundamental problem that killed the ferrofluid approach:
**phase diffusion = 0%** because glass is a solid. No Brownian motion, no
nanoparticle rotational diffusion. Q factors are 10–1000× higher, SNR goes
from -6.5 dB to ~99 dB, and everything can be built from Amazon/hardware store parts.

Historical precedent: mercury delay line memory (UNIVAC I, 1951) stored data
as acoustic waves in a physical medium. WCFOMA glass is the modern, multi-mode,
spectral-encoding version — 75 years of enabling technology later.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from simulations.glass_resonator import (
    glass_database, RodGeometry, compute_mode_spectrum, thermal_stability,
    compute_snr, rayleigh_perturbation, Perturbation, perturbation_capacity,
    associative_recall, technology_comparison, bill_of_materials,
    experiment_plan, glass_resonator_summary,
)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print("✅ Glass resonator module loaded")

## 1. Glass Material Comparison

Four glass types ranked by acoustic quality factor (Q). Higher Q = sharper resonances = more information per mode. Borosilicate (Pyrex) is the sweet spot: Q=10,000, low thermal drift, and $0.80/rod.

In [ ]:
db = glass_database()

# Property table
print(f"{'Glass Type':<30} {'v (m/s)':<10} {'ρ (kg/m³)':<10} {'Q':<10} {'α (ppm/K)':<10} {'Cost'}")
print("─" * 90)
for key, g in db.items():
    print(f"{g.name:<30} {g.v_longitudinal:<10.0f} {g.density:<10.0f} {g.Q_acoustic:<10,.0f} {g.alpha_thermal*1e6:<10.1f} {g.cost_note}")

# Bar charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
names = [g.name.split('(')[0].strip() for g in db.values()]
Qs = [g.Q_acoustic for g in db.values()]
alphas = [g.alpha_thermal * 1e6 for g in db.values()]

colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
ax1.barh(names, Qs, color=colors)
ax1.set_xscale('log')
ax1.set_xlabel('Q Factor')
ax1.set_title('Acoustic Quality Factor (higher = better)')
for i, q in enumerate(Qs):
    ax1.text(q * 1.2, i, f'{q:,.0f}', va='center', fontweight='bold')

ax2.barh(names, alphas, color=colors)
ax2.set_xlabel('Thermal Expansion (ppm/K)')
ax2.set_title('Thermal Drift Coefficient (lower = better)')
for i, a in enumerate(alphas):
    ax2.text(a + 0.2, i, f'{a:.1f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Mode Spectrum — What a Glass Rod Sounds Like

Longitudinal acoustic modes of a 6mm × 150mm borosilicate rod. Each vertical line is a resonant frequency where the rod "rings." The fundamental is ~18.3 kHz (ultrasonic — above hearing). Each mode is an independent information channel.

In [ ]:
rod = RodGeometry(length=0.15, diameter=6e-3, glass_type="borosilicate")
spec = compute_mode_spectrum(rod, n_modes=50)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [2, 1]})

# Mode spectrum as stem plot
ax1.stem(spec.frequencies / 1000, np.ones(50), linefmt='C0-', markerfmt='C0o', basefmt='k-')
ax1.set_xlabel('Frequency (kHz)')
ax1.set_ylabel('Mode')
ax1.set_title(f'Longitudinal Modes: 6mm × 150mm Borosilicate Rod (f₁ = {spec.f_fundamental:.0f} Hz)')
ax1.set_xlim(0, spec.frequencies[49] / 1000 * 1.02)

# Coherence time vs mode number
ax2.semilogy(spec.mode_numbers, spec.coherence_times * 1000, 'C1-o', markersize=3)
ax2.set_xlabel('Mode Number')
ax2.set_ylabel('Coherence Time (ms)')
ax2.set_title(f'Ring-down Time per Mode (Q = {spec.glass.Q_acoustic:,.0f})')
ax2.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='1 ms')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nFundamental: {spec.f_fundamental:.1f} Hz ({spec.f_fundamental/1000:.2f} kHz)")
print(f"Mode spacing: {spec.mode_spacing:.1f} Hz")
print(f"Rod mass: {spec.mass*1000:.2f} g")
print(f"Mode 1 coherence: {spec.coherence_times[0]*1000:.1f} ms ({spec.coherence_times[0]:.3f} s)")
print(f"Mode 1 linewidth: {spec.linewidths[0]:.2f} Hz")
print(f"Mode 50 frequency: {spec.frequencies[49]/1000:.1f} kHz")
print(f"Mode 50 coherence: {spec.coherence_times[49]*1000:.1f} ms")

## 3. The Killer Chart: SNR Comparison — Glass vs. Ferrofluid

This is the single most important result in the entire WCFOMA project. Phase diffusion in ferrofluid killed SNR; a solid glass rod eliminates it entirely.

In [ ]:
# SNR for all glass types vs ferrofluid
snr_results = {}
for gtype in glass_database():
    r = RodGeometry(glass_type=gtype)
    snr_results[gtype] = compute_snr(r, A_drive=1e-9)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# SNR bar chart
labels = [snr_results[k].glass_type.replace('_', ' ').title() for k in snr_results]
snrs = [snr_results[k].SNR_dB for k in snr_results]
labels += ['Ferrofluid\n(baseline)', 'Ferrofluid\n(mitigated)']
snrs += [-6.5, 13.5]
colors = ['#2ecc71', '#27ae60', '#1abc9c', '#16a085', '#e74c3c', '#e67e22']

bars = ax1.bar(labels, snrs, color=colors, edgecolor='black', linewidth=0.5)
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.set_ylabel('SNR (dB)')
ax1.set_title('Signal-to-Noise Ratio at 1nm Drive\n(higher = more bits per mode)')
for bar, val in zip(bars, snrs):
    ypos = max(val, 0) + 2
    ax1.text(bar.get_x() + bar.get_width()/2, ypos, f'{val:.1f} dB',
             ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.set_ylim(-20, 120)

# Phase diffusion comparison
phase_labels = ['Glass\n(any type)', 'Ferrofluid\n(any config)']
phase_vals = [0.0, 77.5]
phase_colors = ['#2ecc71', '#e74c3c']
ax2.bar(phase_labels, phase_vals, color=phase_colors, edgecolor='black',
        linewidth=0.5, width=0.5)
ax2.set_ylabel('Phase Diffusion (% of noise budget)')
ax2.set_title('Phase Diffusion — The Ferrofluid Killer\n(lower = better)')
for i, v in enumerate(phase_vals):
    ax2.text(i, v + 2, f'{v}%', ha='center', fontweight='bold', fontsize=14)
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

# Print detailed SNR for borosilicate
snr_b = snr_results['borosilicate']
print(f"\n{'='*50}")
print(f"Borosilicate SNR Analysis (1nm drive, 300K)")
print(f"{'='*50}")
print(f"Drive amplitude:    {snr_b.A_drive*1e9:.1f} nm")
print(f"Thermal amplitude:  {snr_b.A_thermal*1e15:.2f} fm (femtometers)")
print(f"SNR (linear):       {snr_b.SNR_linear:.2e}")
print(f"SNR (dB):           {snr_b.SNR_dB:.1f} dB")
print(f"Bits/mode (Shannon): {snr_b.bits_per_mode:.1f}")
print(f"Phase diffusion:    {snr_b.phase_diffusion_fraction:.0f}%")
print(f"\n🔑 Glass advantage over ferrofluid baseline: {snr_b.SNR_dB - (-6.5):.0f} dB")

## 4. Thermal Stability — Thousands of Usable Modes

The number of modes that remain resolvable under temperature drift ΔT. Glass's combination of low thermal expansion and high Q gives thousands of usable modes where ferrofluid gave ~10.

In [ ]:
delta_Ts = np.linspace(0.1, 10, 50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for gtype, color in zip(['soda_lime', 'borosilicate', 'fused_silica', 'lead_glass'],
                         ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']):
    modes = []
    bits = []
    for dT in delta_Ts:
        r = RodGeometry(glass_type=gtype)
        stab = thermal_stability(r, delta_T=dT)
        modes.append(stab.max_safe_modes)
        bits.append(stab.total_bits)
    label = glass_database()[gtype].name.split('(')[0].strip()
    ax1.semilogy(delta_Ts, modes, '-', color=color, linewidth=2, label=label)
    ax2.semilogy(delta_Ts, np.array(bits) / 8, '-', color=color, linewidth=2, label=label)

# Add ferrofluid reference lines
ax1.axhline(y=10, color='gray', linestyle='--', alpha=0.7, label='Ferrofluid (mitigated): 10')
ax1.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Ferrofluid (baseline): 0')

ax1.set_xlabel('Temperature Variation ΔT (K)')
ax1.set_ylabel('Usable Modes')
ax1.set_title('Thermally Resolvable Modes vs Temperature Stability')
ax1.legend(fontsize=8)

ax2.set_xlabel('Temperature Variation ΔT (K)')
ax2.set_ylabel('Total Capacity (bytes)')
ax2.set_title('Dynamic Storage Capacity per Rod')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Key numbers at ΔT=1K
print("\nCapacity at ΔT = ±1K:")
print(f"{'Glass':<20} {'Modes':<10} {'Bits/mode':<12} {'Total bits':<15} {'Bytes'}")
print("─" * 70)
for gtype in ['soda_lime', 'borosilicate', 'fused_silica', 'lead_glass']:
    r = RodGeometry(glass_type=gtype)
    stab = thermal_stability(r, delta_T=1.0)
    name = glass_database()[gtype].name.split('(')[0].strip()
    print(f"{name:<20} {stab.max_safe_modes:<10,} {stab.bits_per_mode:<12.1f} {stab.total_bits:<15,.0f} {stab.total_bytes:,.0f}")
print(f"{'Ferrofluid (mitig.)':<20} {'10':<10} {'2.28':<12} {'22.8':<15} {'2.9'}")

## 5. Perturbation Encoding — Writing Data into Glass

Attach a small mass (wax, tape, solder) at position x along the rod. The Rayleigh perturbation formula gives the frequency shift of each mode:

$$\frac{\Delta f_n}{f_n} = -\frac{\Delta m}{2M} \sin^2\!\left(\frac{n\pi x}{L}\right)$$

Different positions create different shift patterns — a unique **spectral fingerprint** that IS the stored data.

In [ ]:
rod = RodGeometry()
n_modes = 20

# Three distinct patterns
patterns = {
    "Pattern A: wax at L/4": [Perturbation(rod.length * 0.25, 0.1e-3, "wax@L/4")],
    "Pattern B: wax at L/3": [Perturbation(rod.length * 0.33, 0.1e-3, "wax@L/3")],
    "Pattern C: two blobs": [
        Perturbation(rod.length * 0.20, 0.05e-3, "wax@L/5"),
        Perturbation(rod.length * 0.70, 0.15e-3, "wax@7L/10"),
    ],
}

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
colors = ['#e74c3c', '#2ecc71', '#3498db']

for idx, (label, perturbs) in enumerate(patterns.items()):
    result = rayleigh_perturbation(rod, perturbs, n_modes=n_modes)
    ax = axes[idx]
    bars = ax.bar(result.mode_numbers, result.shift_hz,
                  color=[colors[idx] if d else '#ddd' for d in result.detectable],
                  edgecolor='black', linewidth=0.3)
    ax.set_ylabel('Δf (Hz)')
    ax.set_title(label)
    n_det = sum(result.detectable)
    ax.text(0.98, 0.85, f'{n_det}/{n_modes} modes detectable',
            transform=ax.transAxes, ha='right', fontsize=10,
            bbox=dict(boxstyle='round', facecolor=colors[idx], alpha=0.2))

axes[-1].set_xlabel('Mode Number')
fig.suptitle('Spectral Fingerprints from Different Perturbation Patterns', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Show that patterns are distinguishable via correlation
sigs = []
sig_labels = []
for label, perturbs in patterns.items():
    result = rayleigh_perturbation(rod, perturbs, n_modes=n_modes)
    sigs.append(result.signature)
    sig_labels.append(label.split(':')[0])

print("\nCross-correlation matrix (1.0 = identical, 0.0 = orthogonal):")
print(f"{'':>12}", end='')
for l in sig_labels:
    print(f"{l:>12}", end='')
print()
for i, li in enumerate(sig_labels):
    print(f"{li:>12}", end='')
    for j in range(len(sigs)):
        corr = np.dot(sigs[i], sigs[j]) / (np.linalg.norm(sigs[i]) * np.linalg.norm(sigs[j]))
        print(f"{corr:>12.3f}", end='')
    print()

## 6. Associative Recall — Content-Addressable Memory

Drive an array of rods with a query pattern. The rod whose perturbation pattern best matches the query resonates most strongly. This is **physical Hopfield recall** via acoustic interference.

In [ ]:
# Create 5 stored patterns (5 rods with different perturbations)
rng = np.random.default_rng(42)
rod = RodGeometry()
n_modes = 15
stored_sigs = []
rod_labels = []

positions = [0.15, 0.25, 0.35, 0.50, 0.75]  # fraction of L
for i in range(5):
    perturbs = [
        Perturbation(rod.length * positions[i], 0.1e-3 * (1 + 0.5 * i)),
        Perturbation(rod.length * (1 - positions[i]), 0.05e-3 * (i + 1)),
    ]
    result = rayleigh_perturbation(rod, perturbs, n_modes=n_modes)
    stored_sigs.append(result.signature)
    rod_labels.append(f"Rod {i+1}")

# Query = noisy version of Rod 3
query_clean = stored_sigs[2].copy()
noise = rng.normal(0, 0.15, n_modes)
query_noisy = query_clean + noise

# Run recall
recall = associative_recall(query_noisy, stored_sigs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Correlation scores
bar_colors = ['#95a5a6'] * 5
bar_colors[recall.best_match_index] = '#2ecc71'
ax1.bar(rod_labels, recall.correlations, color=bar_colors, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Correlation with Query')
ax1.set_title(f'Associative Recall — Query matches Rod {recall.best_match_index + 1}')
ax1.set_ylim(0, 1.1)
for i, c in enumerate(recall.correlations):
    ax1.text(i, c + 0.02, f'{c:.3f}', ha='center', fontweight='bold' if i == recall.best_match_index else 'normal')

# Show query vs best match signature
ax2.plot(range(1, n_modes + 1), query_noisy / np.max(np.abs(query_noisy)),
         'ro-', label='Query (noisy)', markersize=5, alpha=0.7)
ax2.plot(range(1, n_modes + 1), stored_sigs[recall.best_match_index],
         'g^-', label=f'Rod {recall.best_match_index + 1} (stored)', markersize=5)
ax2.set_xlabel('Mode Number')
ax2.set_ylabel('Normalized Signature')
ax2.set_title('Query vs Best Match Spectral Fingerprint')
ax2.legend()

plt.tight_layout()
plt.show()
print(f"\n✅ Best match: Rod {recall.best_match_index + 1} (correlation = {recall.best_match_correlation:.4f})")

## 7. Technology Comparison & Bill of Materials

In [ ]:
# Technology comparison
entries = technology_comparison()
print("WCFOMA Technology Comparison")
print("=" * 100)
print(f"{'Technology':<28} {'SNR(dB)':<10} {'Bits/mode':<10} {'Modes@1K':<10} {'Q':<10} {'Phase Diff':<12} {'Cost'}")
print("─" * 100)
for e in entries:
    print(f"{e.name:<28} {e.SNR_dB:<10.1f} {e.bits_per_mode:<10.1f} {e.modes_at_1K:<10,} "
          f"{e.Q_factor:<10,.0f} {e.phase_diffusion_pct:<12.1f} {e.prototype_cost}")

print("\n")

# Bill of materials
items, total = bill_of_materials()
print("GARAGE PROTOTYPE BILL OF MATERIALS")
print("=" * 75)
print(f"{'#':<4} {'Item':<45} {'Cost':<10} {'Source'}")
print("─" * 75)
for i, item in enumerate(items, 1):
    print(f"{i:<4} {item.item:<45} ${item.cost_usd:<9.2f} {item.source}")
print("─" * 75)
print(f"{'':4} {'TOTAL':<45} ${total:<9.2f}")
print(f"\n💰 Complete prototype for ${total:.0f} — everything from Amazon/hardware store")

## 8. Garage Experiment Plan

Five experiments, ordered from simplest to most ambitious. EXP-G01 validates the physics in an afternoon. If it works, each subsequent experiment builds toward a working content-addressable memory.

In [ ]:
plan = experiment_plan()
for exp in plan:
    print(f"{'='*70}")
    print(f"  {exp['id']}: {exp['name']}")
    print(f"{'='*70}")
    print(f"  Description:   {exp['description']}")
    print(f"  Setup:         {exp['setup'][:100]}...")
    print(f"  Predicted:     {exp['predicted']}")
    print(f"  Success:       {exp['success_criterion'][:80]}...")
    print(f"  Kill switch:   {exp['kill_criterion'][:80]}...")
    print()

## 9. Full Summary

In [ ]:
print(glass_resonator_summary())

# Encoding capacity
cap10 = perturbation_capacity(n_perturbations=10, position_resolution=100, mass_levels=8)
cap50 = perturbation_capacity(n_perturbations=50, position_resolution=100, mass_levels=8)
print(f"\nNon-volatile perturbation encoding:")
print(f"  10 wax blobs: {cap10.total_bits:.0f} bits ({cap10.total_bytes:.0f} bytes)")
print(f"  50 wax blobs: {cap50.total_bits:.0f} bits ({cap50.total_bytes:.0f} bytes)")

print(f"""
{'='*60}
BOTTOM LINE
{'='*60}
✅ Phase diffusion eliminated (0% vs 77.5%)
✅ SNR increased by ~105 dB
✅ Modes increased from 10 to 9,380+
✅ Prototype cost: $63 (vs $500-1000)
✅ All materials from Amazon/hardware store
✅ No exotic nanoparticles, no cleanroom
✅ Historical precedent (delay line memory, 1951)
✅ Can be built and tested in a weekend

This is the path to a real prototype.
""")